# 5. CrewAI — Multi-Agent Collaboration

## Overview

LangChain gives you one agent with many tools. LangGraph gives you a graph you control.
**CrewAI** takes a third position: model the work the way a *team* would do it. You define
people — a researcher, an analyst, a writer — give each one a role and a goal, hand the team a
list of tasks, and let them pass work between themselves.

The team metaphor fits parts of healthcare administration unusually well, because the work is
already organised that way. A benefits question that reaches a member goes through a benefits
researcher, a coverage analyst and a communications writer in most plans. A new member-facing
tool goes past privacy, clinical governance and operations before it ships. Those are real
hand-offs between real specialists, and CrewAI lets you write them down almost literally.

In this notebook you will:

- Point CrewAI at **Amazon Bedrock**
- Write effective **roles, goals and backstories**
- Chain tasks so one agent's output becomes another's input
- Give agents **custom tools**
- Run a **hierarchical** crew where a manager agent delegates
- Capture **structured output** and inspect token usage


## Concepts

| Concept | Meaning |
|---|---|
| **Agent** | A role-playing worker: `role`, `goal`, `backstory`, optional `tools` |
| **Task** | A unit of work: `description`, `expected_output`, assigned to an agent |
| **Crew** | The team plus the task list plus a process |
| **Process** | `sequential` (tasks in order) or `hierarchical` (a manager delegates) |
| **Context** | Passing one task's output into a later task |
| **Delegation** | An agent asking a colleague for help mid-task |

> **Where CrewAI belongs in this domain — and where it does not.**
>
> Notice what this notebook does *not* do: it never puts a crew in front of a member, and it
> never lets a crew make a coverage determination. Every crew here produces **internal drafts
> and reviews that a human then approves**.
>
> That is deliberate. CrewAI's strength is readable collaboration; its weakness is that the path
> varies between runs and there is no state you can branch on or replay. For anything a member
> receives or a regulator inspects, use the LangGraph pattern from notebook 4. Use a crew where
> a human is the last step.
>
> All plan data below is invented for training.

#### 0. Setup

CrewAI reaches Bedrock through LiteLLM, which uses the standard boto3 credential chain — so on
SageMaker it picks up your **execution role** with no configuration at all.

The one thing it does need is the region under the name it expects: LiteLLM looks for
**`AWS_REGION_NAME`**, not `AWS_REGION`, so the cell below sets both.

In [18]:
# ---------------------------------------------------------------------------
# SageMaker Studio setup
#
# There is no .env file and no access key in this lab. boto3 automatically
# picks up short-lived credentials from the SageMaker execution role attached
# to this space, which is both simpler and safer than a long-lived key pair.
# ---------------------------------------------------------------------------
import os
import boto3

# Leave this as None to use the region this SageMaker space is running in.
# Set it ONLY if you enabled Bedrock model access in a different region.
BEDROCK_REGION = None

AWS_REGION = BEDROCK_REGION or boto3.session.Session().region_name or "us-east-1"
CHAT_MODEL_ID = os.environ.get("BEDROCK_CHAT_MODEL_ID", "amazon.nova-pro-v1:0")

session = boto3.Session(region_name=AWS_REGION)
EXECUTION_ROLE_ARN = session.client("sts").get_caller_identity()["Arn"]

# CrewAI / LiteLLM look for these exact names
os.environ["AWS_REGION_NAME"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

from crewai import Agent, Task, Crew, Process, LLM

bedrock_llm = LLM(
    model=f"bedrock/{CHAT_MODEL_ID}",
    temperature=0.3,
    max_tokens=2048,
)

print("Region         :", AWS_REGION)
print("Execution role :", EXECUTION_ROLE_ARN)
print("CrewAI LLM configured for bedrock/" + CHAT_MODEL_ID)

Region         : us-east-1
Execution role : arn:aws:sts::442042521384:assumed-role/AmazonSageMaker-ExecutionRole-20260807T102279/SageMaker
CrewAI LLM configured for bedrock/amazon.nova-pro-v1:0


In [20]:
# Smoke test - confirm CrewAI can reach Bedrock before building a crew
print(bedrock_llm.call("Reply with exactly: connection ok"))

connection ok

/home/sagemaker-user/lanchain-healthcare-la/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

---
## 1. A single agent and a single task

Start with the smallest possible crew: one agent, one task. Everything else in CrewAI is a
repetition of this pattern.

#### 1.1. Roles and tasks

An agent is defined by three fields, and each one does real work:

- **`role`** — the job title. Keep it specific: "Medicare Benefits Researcher", not "Helper".
- **`goal`** — what success looks like for this agent, in one sentence.
- **`backstory`** — the experience that shapes *how* it works. This is the highest-leverage
  field; it is where you encode standards, biases and method.

A task is defined by:

- **`description`** — what to do, including any `{placeholders}` filled at kickoff
- **`expected_output`** — the shape of the deliverable. Be concrete. Vague expected output is
  the single most common cause of disappointing crew results.

In [3]:
researcher = Agent(
    role="Medicare Benefits Researcher",
    goal="Find accurate, specific, plainly-stated facts about how {topic} works",
    backstory=(
        "You spent eight years in a health plan's benefits configuration team. You have "
        "watched members be harmed by confident, imprecise answers, so you never state a "
        "coverage rule without naming its conditions and exceptions. You distinguish sharply "
        "between what Original Medicare covers and what varies by plan. When you are unsure, "
        "you say so explicitly rather than filling the gap with a plausible guess. "
        "You never give clinical advice."
    ),
    llm=bedrock_llm,
    verbose=True,
    allow_delegation=False,
)

research_task = Task(
    description=(
        "Research how {topic} works under Medicare. Cover: what it is, who it applies to, "
        "the conditions and exceptions, what the member typically pays, and what varies by "
        "plan type."
    ),
    expected_output=(
        "A structured brief with exactly five sections: WHAT IT IS, WHO IT APPLIES TO, "
        "CONDITIONS AND EXCEPTIONS, MEMBER COST, WHAT VARIES BY PLAN. "
        "Each section 3-5 bullet points. No clinical advice. No marketing language."
    ),
    agent=researcher,
)

crew = Crew(agents=[researcher], tasks=[research_task], verbose=True)
print("Crew ready")

Crew ready


#### 1.2. Kick it off

`kickoff()` runs the crew. The `inputs` dictionary fills every `{placeholder}` in the agent
and task definitions.

In [4]:
result = crew.kickoff(inputs={"topic": "the Part B late enrolment penalty"})

print("\n" + "=" * 70)
print(result.raw)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7d9f9b92-d336-4f79-83c5-8370bece032a                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medicare Benefits Researcher                                                                            │
│                                                                                                                 │
│  Task: Research how the Part B late enrolment penalty works under Medicare. Cover: what it is, who it applies   │
│  to, the conditions and exceptions, what the member typically pays, and what varies by plan type.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medicare Benefits Researcher                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **WHAT IT IS:**                                                                                                │
│  - The Part B late enrollment penalty is a financial charge applied to individuals who do not sign up for       │
│  Medicare Part B when they are first eligible.                                                                  │
│  - This penalty is added to the individual's monthly Part B premium.                                            │
│  - The purpose of the penalty is to encourage timely enrollment in Medicare Part B to avoid increased costs to  │
│  the Medicare program.                                                                                          │
│                                                                                                                 │
│  **WHO IT APPLIES TO:**                                                                                         │
│  - It applies to individuals who are eligible for Medicare Part B but do not enroll during their Initial        │
│  Enrollment Period (IEP).                                                                                       │
│  - The IEP is a 7-month window that begins three months before the month an individual turns 65 and ends three  │
│  months after that month.                                                                                       │
│  - It also applies to individuals who are eligible for but do not enroll in Part B while they are still         │
│  working and covered under an employer group health plan.                                                       │
│                                                                                                                 │
│  **CONDITIONS AND EXCEPTIONS:**                                                                                 │
│  - If an individual has group health coverage through current employment or their spouse’s current employment   │
│  when they become eligible for Medicare, they can delay enrolling in Part B without penalty as long as the      │
│  employer has 20 or more employees.                                                                             │
│  - If an individual loses their employer coverage after enrolling in Medicare, they can enroll in Part B        │
│  during a Special Enrollment Period without penalty.                                                            │
│  - Individuals who qualify for a Special Enrollment Period due to other specific circumstances (like moving     │
│  out of the service area of their Medicare Advantage Plan) may also avoid the penalty.                          │
│                                                                                                                 │
│  **MEMBER COST:**                                                                                               │
│  - The penalty is 10% for every 12-month period that an individual could have had Part B, but did not enroll.   │
│  - This 10% increase is added to the standard Part B premium for as long as the individual has Part B.          │
│  - For example, if an individual waits three years to enroll after becoming eligible, their premium will be     │
│  30% higher than the standard Part B premium.                                                                   │
│                                                        

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 2c4eaf4c-3da0-4e40-a1e9-cb144955f0f9                                                                     │
│  Agent: Medicare Benefits Researcher                                                                            │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7d9f9b92-d336-4f79-83c5-8370bece032a                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: **WHAT IT IS:**                                                                                  │
│  - The Part B late enrollment penalty is a financial charge applied to individuals who do not sign up for       │
│  Medicare Part B when they are first eligible.                                                                  │
│  - This penalty is added to the individual's monthly Part B premium.                                            │
│  - The purpose of the penalty is to encourage timely enrollment in Medicare Part B to avoid increased costs to  │
│  the Medicare program.                                                                                          │
│                                                                                                                 │
│  **WHO IT APPLIES TO:**                                                                                         │
│  - It applies to individuals who are eligible for Medicare Part B but do not enroll during their Initial        │
│  Enrollment Period (IEP).                                                                                       │
│  - The IEP is a 7-month window that begins three months before the month an individual turns 65 and ends three  │
│  months after that month.                                                                                       │
│  - It also applies to individuals who are eligible for but do not enroll in Part B while they are still         │
│  working and covered under an employer group health plan.                                                       │
│                                                                                                                 │
│  **CONDITIONS AND EXCEPTIONS:**                                                                                 │
│  - If an individual has group health coverage through current employment or their spouse’s current employment   │
│  when they become eligible for Medicare, they can delay enrolling in Part B without penalty as long as the      │
│  employer has 20 or more employees.                                                                             │
│  - If an individual loses their employer coverage after enrolling in Medicare, they can enroll in Part B        │
│  during a Special Enrollment Period without penalty.                                                            │
│  - Individuals who qualify for a Special Enrollment Period due to other specific circumstances (like moving     │
│  out of the service area of their Medicare Advantage Plan) may also avoid the penalty.                          │
│                                                                                                                 │
│  **MEMBER COST:**                                                                                               │
│  - The penalty is 10% for every 12-month period that an individual could have had Part B, but did not enroll.   │
│  - This 10% increase is added to the standard Part B premium for as long as the individual has Part B.          │
│  - For example, if an individual waits three years to enroll after becoming eligible, their premium will be     │
│  30% higher than the standard Part B premium.         

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

╭───────────────────────────────────────── 🔍 Execution Trace Generated ──────────────────────────────────────────╮
│                                                                                                                 │
│  🎉 Your First CrewAI Execution Trace is Ready!                                                                 │
│                                                                                                                 │
│  View your execution details here:                                                                              │
│  https://app.crewai.com/crewai_plus/ephemeral_trace_batches/f7296c5c-97ca-416a-9f3c-e953c27472d1?access_code=T  │
│  RACE-f54e49377f                                                                                                │
│                                                                                                                 │
│  This trace shows:                                                                                              │
│  • Agent decisions and interactions                                                                             │
│  • Task execution timeline                                                                                      │
│  • Tool usage and results                                                                                       │
│  • LLM calls and responses                                                                                      │
│                                                                                                                 │
│  ✅ Tracing has been enabled for future runs! (CREWAI_TRACING_ENABLED=true added to .env)                       │
│  You can also add tracing=True to your Crew(tracing=True) / Flow(tracing=True) for more control.                │
│                                                                                                                 │
│  📝 Note: This link will expire in 24 hours.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


**WHAT IT IS:**
- The Part B late enrollment penalty is a financial charge applied to individuals who do not sign up for Medicare Part B when they are first eligible.
- This penalty is added to the individual's monthly Part B premium.
- The purpose of the penalty is to encourage timely enrollment in Medicare Part B to avoid increased costs to the Medicare program.

**WHO IT APPLIES TO:**
- It applies to individuals who are eligible for Medicare Part B but do not enroll during their Initial Enrollment Period (IEP).
- The IEP is a 7-month window that begins three months before the month an individual turns 65 and ends three months after that month.
- It also applies to individuals who are eligible for but do not enroll in Part B while they are still working and covered under an employer group health plan.

**CONDITIONS AND EXCEPTIONS:**
- If an individual has group health coverage through current employment or their spouse’s current employment when they become eligible for Medicare, the

> **Read that output critically.** It is fluent, well-organised and **completely ungrounded** —
> the researcher had no tools and no source document, so every fact came from model memory.
> Compare it to the cited answers from notebook 3.
>
> This is the central trade in CrewAI: you get readable collaboration quickly, and you have to
> add grounding yourself. Section 3 attaches tools; a production version would attach a
> retrieval tool pointed at your own plan documents.

---
## 2. A multi-agent crew

Now the real pattern: three specialists, each doing the part they are good at, passing work
down the line. This mirrors how a member communication is actually produced.

```
  Researcher  ->  Coverage Analyst  ->  Communications Writer
   (finds)          (judges)                (translates)
```

The `context` parameter on a task is what makes this work — it feeds earlier task outputs into
the later task's prompt.

In [5]:
analyst = Agent(
    role="Coverage Analyst",
    goal="Turn benefits research about {topic} into a defensible position on what members should be told",
    backstory=(
        "You review benefits language for a health plan before it goes out. You have seen "
        "appeals and complaints trace back to a single over-confident sentence in a member "
        "letter. You always separate what is universally true from what depends on the "
        "member's plan, and you always name the case where the general rule does not hold."
    ),
    llm=bedrock_llm,
    verbose=True,
    allow_delegation=False,
)

writer = Agent(
    role="Member Communications Writer",
    goal="Produce a clear one-page explanation a member can read once and act on",
    backstory=(
        "You write for people who are stressed, possibly unwell, and reading on a phone. "
        "You write at a sixth-grade reading level without being condescending. You lead with "
        "what the member needs to do. You state every deadline as a date or a number of days, "
        "never as 'promptly'. You never give clinical advice, and you always tell the member "
        "how to reach a human."
    ),
    llm=bedrock_llm,
    verbose=True,
    allow_delegation=False,
)

In [6]:
task_research = Task(
    description=(
        "Research how {topic} works under Medicare. Cover the rule, its conditions, the "
        "exceptions, typical member cost, and what varies by plan type."
    ),
    expected_output="A factual brief of 250-350 words with clearly labelled sections.",
    agent=researcher,
)

task_analyse = Task(
    description=(
        "Using the research provided, decide what a member facing {topic} should actually be "
        "told. Separate what is always true from what depends on their plan. Name the "
        "situations where the general rule does not apply."
    ),
    expected_output=(
        "A position with three parts: ALWAYS TRUE, DEPENDS ON PLAN, and EXCEPTIONS TO FLAG. "
        "200 words maximum."
    ),
    agent=analyst,
    context=[task_research],       # <-- receives the researcher's output
)

task_write = Task(
    description=(
        "Turn the research and the analysis into a one-page member explanation of {topic}."
    ),
    expected_output=(
        "A markdown document with: a one-sentence summary in bold, a 'What this means for you' "
        "section, a 'What you need to do' numbered list with any deadline stated explicitly, "
        "and a closing line telling the member how to reach a person. "
        "Sixth-grade reading level. No clinical advice."
    ),
    agent=writer,
    context=[task_research, task_analyse],   # <-- receives both
)

comms_crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[task_research, task_analyse, task_write],
    process=Process.sequential,
    verbose=True,
)
print("Three-agent crew ready")

Three-agent crew ready


In [7]:
output = comms_crew.kickoff(inputs={"topic": "a claim denied for missing prior authorisation"})

print("\n" + "=" * 70)
print("FINAL DELIVERABLE\n")
print(output.raw)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c267d7f1-37cd-41dc-877c-c104ccb4756c                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medicare Benefits Researcher                                                                            │
│                                                                                                                 │
│  Task: Research how a claim denied for missing prior authorisation works under Medicare. Cover the rule, its    │
│  conditions, the exceptions, typical member cost, and what varies by plan type.                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medicare Benefits Researcher                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Rule:** Under Medicare, a claim may be denied if prior authorization is required but not obtained. This      │
│  rule applies to both Original Medicare (Parts A and B) and Medicare Advantage (Part C) plans, though the       │
│  specifics can vary.                                                                                            │
│                                                                                                                 │
│  **Conditions:** Prior authorization is typically required for certain services, procedures, or medications     │
│  that Medicare deems necessary to ensure they are medically necessary and appropriate. The conditions for       │
│  requiring prior authorization include, but are not limited to, specific types of surgeries, durable medical    │
│  equipment (DME), and certain prescription drugs under Medicare Part D.                                         │
│                                                                                                                 │
│  **Exceptions:** Some services are exempt from prior authorization requirements. For instance, emergency        │
│  services are generally covered without the need for prior authorization. Additionally, some Medicare           │
│  Advantage plans may have different rules regarding which services require prior authorization compared to      │
│  Original Medicare.                                                                                             │
│                                                                                                                 │
│  **Typical Member Cost:** If a claim is denied for missing prior authorization, the member may be responsible   │
│  for the full cost of the service or procedure. In Original Medicare, this could mean paying the entire bill    │
│  out-of-pocket. For Medicare Advantage plans, the cost can vary depending on the plan’s specific rules and      │
│  coverage details.                                                                                              │
│                                                                                                                 │
│  **What Varies by Plan Type:**                                                                                  │
│  - **Original Medicare:** Generally follows standard Medicare guidelines for prior authorization. Members may   │
│  need to pay the full cost if prior authorization is missing.                                                   │
│  - **Medicare Advantage (Part C):** Plans may have additional or different prior authorization requirements.    │
│  Costs can vary widely depending on the plan’s coverage and the specific service denied.                        │
│  - **Medicare Part D:** Prescription drug coverage may require prior authorization for certain medications.     │
│  Denials for missing prior authorization can result in the member paying the full cost of the medication.       │
│                                                                                                                 │
│  In summary, understanding the specific requirements for prior authorization and the potential costs involved   │
│  is crucial for Medicare beneficiaries to avoid claim denials and unexpected expenses. Always check with your   │
│  specific plan for detailed information on their prior 

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 7a6847f6-e824-43c3-bfb4-efe037a1885b                                                                     │
│  Agent: Medicare Benefits Researcher                                                                            │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Coverage Analyst                                                                                        │
│                                                                                                                 │
│  Task: Using the research provided, decide what a member facing a claim denied for missing prior authorisation  │
│  should actually be told. Separate what is always true from what depends on their plan. Name the situations     │
│  where the general rule does not apply.                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: a1842dc4-0fb1-4ff7-9f95-e56713701a27                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/a1842dc4-0fb1-4ff7-9f95-e56713701a27?access_code=TRA │
│ CE-945dae607e                                                                                                   │
│ 🔑 Access Code: TRACE-945dae607e                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── 🔍 Execution Trace Generated ──────────────────────────────────────────╮
│                                                                                                                 │
│  🎉 Your First CrewAI Execution Trace is Ready!                                                                 │
│                                                                                                                 │
│  View your execution details here:                                                                              │
│  https://app.crewai.com/crewai_plus/ephemeral_trace_batches/a1842dc4-0fb1-4ff7-9f95-e56713701a27?access_code=T  │
│  RACE-945dae607e                                                                                                │
│                                                                                                                 │
│  This trace shows:                                                                                              │
│  • Agent decisions and interactions                                                                             │
│  • Task execution timeline                                                                                      │
│  • Tool usage and results                                                                                       │
│  • LLM calls and responses                                                                                      │
│                                                                                                                 │
│  ✅ Tracing has been enabled for future runs! (CREWAI_TRACING_ENABLED=true added to .env)                       │
│  You can also add tracing=True to your Crew(tracing=True) / Flow(tracing=True) for more control.                │
│                                                                                                                 │
│  📝 Note: This link will expire in 24 hours.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL DELIVERABLE

---

**Your claim was denied because you did not get prior authorization.**

### What this means for you

If you need certain services, procedures, or medications, Medicare may require you to get prior authorization. This means you need to get approval before you get the service or medication. If you do not get this approval, your claim may be denied. This applies to both Original Medicare (Parts A and B) and Medicare Advantage (Part C) plans. If your claim is denied, you may have to pay the full cost of the service or procedure.

### What you need to do

1. **Check your plan’s rules:** Look at your Medicare plan documents to see which services need prior authorization. 
2. **Get prior authorization:** If a service needs prior authorization, make sure to get it before you get the service or medication. 
3. **Pay the cost if needed:** If your claim was denied, you may need to pay the full cost of the service or procedure. 
4. **Appeal if you think it’s wrong:** If yo

#### 2.1. Inspect each agent's contribution

In [8]:
for i, task_output in enumerate(output.tasks_output, 1):
    print("=" * 70)
    print(f"TASK {i} - agent: {task_output.agent}")
    print(task_output.raw[:400].strip())
    print()

print("TOKEN USAGE:", output.token_usage)

TASK 1 - agent: Medicare Benefits Researcher
**Rule:** Under Medicare, a claim may be denied if prior authorization is required but not obtained. This rule applies to both Original Medicare (Parts A and B) and Medicare Advantage (Part C) plans, though the specifics can vary.

**Conditions:** Prior authorization is typically required for certain services, procedures, or medications that Medicare deems necessary to ensure they are medically ne

TASK 2 - agent: Coverage Analyst
ALWAYS TRUE: If prior authorization is required by Medicare for a service, procedure, or medication and it is not obtained, your claim may be denied. This applies to both Original Medicare (Parts A and B) and Medicare Advantage (Part C) plans. In such cases, you may be responsible for the full cost of the service or procedure.

DEPENDS ON PLAN: The specific services, procedures, or medications tha

TASK 3 - agent: Member Communications Writer
---

**Your claim was denied because you did not get prior authorization.

---
## 3. Giving agents tools

Agents without tools can only reason over what is already in the prompt. Tools let them act.
CrewAI's `@tool` decorator works like LangChain's: the name and docstring tell the agent when
to use it.

This is how you stop a crew inventing dollar amounts.

In [9]:
from crewai.tools import tool


@tool("Member Cost Estimator")
def member_cost_estimator(billed_amount: float, deductible_remaining: float,
                          coinsurance_pct: float = 20.0) -> str:
    """Estimate what a member owes under a deductible-plus-coinsurance plan.

    billed_amount is the allowed amount in USD, deductible_remaining is the unmet portion
    of the annual deductible, and coinsurance_pct is the member's share after the deductible.
    Returns a sample estimate for training, not a coverage determination.
    """
    to_deductible = min(billed_amount, max(deductible_remaining, 0))
    after_deductible = billed_amount - to_deductible
    coinsurance = round(after_deductible * coinsurance_pct / 100, 2)
    total = round(to_deductible + coinsurance, 2)
    return (
        f"Allowed ${billed_amount}: ${to_deductible} to deductible, "
        f"${coinsurance} coinsurance ({coinsurance_pct}% of ${after_deductible}). "
        f"Estimated member responsibility ${total}."
    )


@tool("Prior Authorisation Checker")
def prior_auth_checker(service: str) -> str:
    """Check whether a service requires prior authorisation under the sample plan.

    Valid services: mri, ct_scan, pet_scan, inpatient_admission, outpatient_surgery,
    physical_therapy, durable_medical_equipment, primary_care_visit, lab_work.
    """
    rules = {
        "mri": "REQUIRED for outpatient MRI",
        "ct_scan": "REQUIRED for outpatient CT",
        "pet_scan": "REQUIRED",
        "inpatient_admission": "REQUIRED except emergency admissions",
        "outpatient_surgery": "REQUIRED for selected procedures",
        "physical_therapy": "Not required for the first 12 visits, then REQUIRED",
        "durable_medical_equipment": "REQUIRED for items over $500",
        "primary_care_visit": "Not required",
        "lab_work": "Not required for in-network labs",
    }
    return rules.get(service.lower().strip().replace(" ", "_"),
                     f"No rule on file for '{service}'")


print("Tools defined")

Tools defined


In [10]:
cost_agent = Agent(
    role="Member Cost Estimator",
    goal="Produce accurate cost estimates using the calculator tool, never by mental arithmetic",
    backstory=(
        "You work in a health plan's cost transparency team. You never estimate a number in "
        "your head when a tool can compute it, and you always show the inputs you used so the "
        "figure can be checked. You state clearly that an estimate is not a guarantee of "
        "payment, because you have seen members plan their finances around one."
    ),
    tools=[member_cost_estimator, prior_auth_checker],
    llm=bedrock_llm,
    verbose=True,
    allow_delegation=False,
)

cost_task = Task(
    description=(
        "A member is scheduled for an outpatient MRI with an allowed amount of $1,842. "
        "They have $400 of their annual deductible remaining and 20% coinsurance. "
        "Check whether the service needs prior authorisation, then calculate what the member "
        "should expect to pay."
    ),
    expected_output=(
        "A short itemised breakdown showing the prior authorisation requirement, the "
        "deductible portion, the coinsurance portion and the total, followed by one line "
        "stating that this is an estimate and not a guarantee of payment."
    ),
    agent=cost_agent,
)

cost_crew = Crew(agents=[cost_agent], tasks=[cost_task], verbose=True)
cost_result = cost_crew.kickoff()

print("\n" + "=" * 70)
print(cost_result.raw)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9f39a642-ad70-4972-8128-ec5b4980ecf3                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Member Cost Estimator                                                                                   │
│                                                                                                                 │
│  Task: A member is scheduled for an outpatient MRI with an allowed amount of $1,842. They have $400 of their    │
│  annual deductible remaining and 20% coinsurance. Check whether the service needs prior authorisation, then     │
│  calculate what the member should expect to pay.                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/sagemaker-user/lanchain-healthcare-la/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Member Cost Estimator                                                                                   │
│                                                                                                                 │
│  Thought: Thought: I need to first check if the outpatient MRI requires prior authorization. Then, I will       │
│  calculate the member's out-of-pocket cost using the Member Cost Estimator.                                     │
│                                                                                                                 │
│  Using Tool: Prior Authorisation Checker                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"service\": \"mri\"}"                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 5994b143-5cec-456d-81a6-2c7ce7613a3f                                                                     │
│  Agent: Member Cost Estimator                                                                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9f39a642-ad70-4972-8128-ec5b4980ecf3                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: The outpatient MRI requires prior authorization. The member will pay $400 towards their          │
│  deductible, $288.40 in coinsurance (20% of $1442, which is the allowed amount minus the deductible), for a     │
│  total estimated member responsibility of $688.40. Please note that this is an estimate and not a guarantee of  │
│  payment.                                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 


The outpatient MRI requires prior authorization. The member will pay $400 towards their deductible, $288.40 in coinsurance (20% of $1442, which is the allowed amount minus the deductible), for a total estimated member responsibility of $688.40. Please note that this is an estimate and not a guarantee of payment.


---
## 4. Orchestrating autonomous agent teams

`Process.sequential` runs tasks in the order you wrote them — you decide the workflow.
`Process.hierarchical` adds a **manager agent** that decides who does what, delegates, reviews
the results, and can send work back.

Two changes are required: set `process=Process.hierarchical`, and supply a `manager_llm`. Do
**not** assign an `agent` to the tasks — that is the manager's job.

The task below is a design review of a proposed member-facing assistant. It is the review a
real health plan would run before shipping anything like what you have built in this lab — and
the architecture described contains a deliberate set of problems for the crew to find.

In [23]:
from crewai import Agent, Task, Crew, Process

# --- Agents
security_reviewer = Agent(
    role="Cloud Security Reviewer",
    goal="Identify security and compliance gaps in AWS architectures",
    backstory=(
        "You audit AWS accounts against CIS benchmarks. You care about IAM least "
        "privilege, encryption at rest and in transit, and data residency."
    ),
    llm=bedrock_llm,
    verbose=True,
)

performance_reviewer = Agent(
    role="Performance Engineer",
    goal="Identify latency and scalability bottlenecks in AWS architectures",
    backstory=(
        "You have debugged production systems at scale. You think in terms of "
        "p99 latency, cold starts, throttling limits and regional round trips."
    ),
    llm=bedrock_llm,
    verbose=True,
)

cost_reviewer = Agent(
    role="FinOps Reviewer",
    goal="Identify cost risks and waste in AWS architectures",
    backstory=(
        "You have cut cloud bills by 40% without changing functionality. You look "
        "for idle capacity, oversized instances and per-token costs that scale badly."
    ),
    llm=bedrock_llm,
    verbose=True,
)

In [24]:
# --- Tasks ---
security_task = Task(
    description=(
        "Review the security and compliance aspects of this architecture:\n\n"
        "'A RAG chatbot for internal HR documents. Documents sit in one public-read "
        "S3 bucket. A Lambda function embeds queries with Titan and calls Claude on "
        "Bedrock in us-east-1. Employees are in India. All Lambda functions share one "
        "IAM role with AdministratorAccess. There is no caching and no request limit.'\n\n"
        "Focus only on IAM, encryption, data exposure, and compliance gaps."
    ),
    expected_output=(
        "SECURITY section only: each finding states the issue, the impact, and the fix. "
        "3-5 bullet points."
    ),
    agent=security_reviewer,
)

performance_task = Task(
    description=(
        "Review the performance and scalability aspects of the same architecture "
        "(see security review in context). Focus on latency, cold starts, throttling, "
        "and the India-to-us-east-1 round trip."
    ),
    expected_output=(
        "PERFORMANCE section only: each finding states the issue, the impact, and the fix. "
        "3-5 bullet points."
    ),
    agent=performance_reviewer,
    context=[security_task],
)

cost_task = Task(
    description=(
        "Review the cost risks of the same architecture (see prior reviews in context). "
        "Focus on token costs at scale, idle Lambda, and missing caching."
    ),
    expected_output=(
        "COST section only: each finding states the issue, the impact, and the fix. "
        "3-5 bullet points."
    ),
    agent=cost_reviewer,
    context=[security_task, performance_task],
)

consolidation_task = Task(
    description=(
        "Consolidate the three specialist reviews (security, performance, cost) from "
        "context into one unified report. Do not add new findings — only synthesise."
    ),
    expected_output=(
        "A consolidated review with three labelled sections: SECURITY, PERFORMANCE, COST. "
        "Each finding must state the issue, the impact, and the fix. "
        "End with the single highest-priority action across all three domains."
    ),
    agent=security_reviewer,
    context=[security_task, performance_task, cost_task],
)

# --- Crew ---
sequential_crew = Crew(
    agents=[security_reviewer, performance_reviewer, cost_reviewer],
    tasks=[security_task, performance_task, cost_task, consolidation_task],
    process=Process.sequential,
    verbose=True,
)

review = sequential_crew.kickoff()

print("\n" + "=" * 70)
print(review.raw)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 401a0195-b913-4b6e-853b-9e8de67992fd                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Cloud Security Reviewer                                                                                 │
│                                                                                                                 │
│  Task: Review the security and compliance aspects of this architecture:                                         │
│                                                                                                                 │
│  'A RAG chatbot for internal HR documents. Documents sit in one public-read S3 bucket. A Lambda function        │
│  embeds queries with Titan and calls Claude on Bedrock in us-east-1. Employees are in India. All Lambda         │
│  functions share one IAM role with AdministratorAccess. There is no caching and no request limit.'              │
│                                                                                                                 │
│  Focus only on IAM, encryption, data exposure, and compliance gaps.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Cloud Security Reviewer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **SECURITY:**                                                                                                  │
│                                                                                                                 │
│  - **Issue:** The Lambda functions use an IAM role with `AdministratorAccess`.                                  │
│    - **Impact:** This violates the principle of least privilege, allowing Lambda functions unnecessary broad    │
│  access across the AWS account, increasing the risk of unauthorized actions and potential data breaches.        │
│    - **Fix:** Create a custom IAM role for the Lambda functions with minimal required permissions, such as      │
│  `s3:GetObject` for the specific S3 bucket, `bedrock:InvokeModel` for Claude on Bedrock, and necessary logging  │
│  permissions.                                                                                                   │
│                                                                                                                 │
│  - **Issue:** The S3 bucket containing HR documents is set to public-read.                                      │
│    - **Impact:** This exposes sensitive HR documents to anyone on the internet, leading to potential data       │
│  breaches and non-compliance with data protection regulations.                                                  │
│    - **Fix:** Remove public access to the S3 bucket. Use bucket policies to restrict access to authorized       │
│  users only, and consider using pre-signed URLs for temporary access if needed.                                 │
│                                                                                                                 │
│  - **Issue:** There is no encryption at rest for the S3 bucket.                                                 │
│    - **Impact:** Sensitive HR documents are stored unencrypted, increasing the risk of data breaches if the     │
│  bucket is compromised.                                                                                         │
│    - **Fix:** Enable server-side encryption (SSE) with AWS KMS for the S3 bucket to ensure data is encrypted    │
│  at rest.                                                                                                       │
│                                                                                                                 │
│  - **Issue:** There is no encryption in transit for communications between Lambda, S3, and Bedrock.             │
│    - **Impact:** Data transmitted between these services is not encrypted, potentially exposing sensitive       │
│  information to interception.                                                                                   │
│    - **Fix:** Ensure that all communications use HTTPS and enable encryption in transit for S3 and Bedrock      │
│  interactions.                                                                                                  │
│                                                                                                                 │
│  - **Issue:** All Lambda functions share one IAM role, which has excessive permissions.                         │
│    - **Impact:** This increases the attack surface and risk of privilege escalation. If one Lambda function is  │
│  compromised, it could lead to unauthorized access to o

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: b032a582-04c0-44ed-87dc-b57bd4d24bbf                                                                     │
│  Agent: Cloud Security Reviewer                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Performance Engineer                                                                                    │
│                                                                                                                 │
│  Task: Review the performance and scalability aspects of the same architecture (see security review in          │
│  context). Focus on latency, cold starts, throttling, and the India-to-us-east-1 round trip.                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 04d86770-77aa-4f38-ade4-0debb4207e72                                                                     │
│  Agent: Cloud Security Reviewer                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 401a0195-b913-4b6e-853b-9e8de67992fd                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: **SECURITY:**                                                                                    │
│                                                                                                                 │
│  - **Issue:** The Lambda functions use an IAM role with `AdministratorAccess`.                                  │
│    - **Impact:** This violates the principle of least privilege, allowing Lambda functions unnecessary broad    │
│  access across the AWS account, increasing the risk of unauthorized actions and potential data breaches.        │
│    - **Fix:** Create a custom IAM role for the Lambda functions with minimal required permissions, such as      │
│  `s3:GetObject` for the specific S3 bucket, `bedrock:InvokeModel` for Claude on Bedrock, and necessary logging  │
│  permissions.                                                                                                   │
│                                                                                                                 │
│  - **Issue:** The S3 bucket containing HR documents is set to public-read.                                      │
│    - **Impact:** This exposes sensitive HR documents to anyone on the internet, leading to potential data       │
│  breaches and non-compliance with data protection regulations.                                                  │
│    - **Fix:** Remove public access to the S3 bucket. Use bucket policies to restrict access to authorized       │
│  users only, and consider using pre-signed URLs for temporary access if needed.                                 │
│                                                                                                                 │
│  - **Issue:** There is no encryption at rest for the S3 bucket.                                                 │
│    - **Impact:** Sensitive HR documents are stored unencrypted, increasing the risk of data breaches if the     │
│  bucket is compromised.                                                                                         │
│    - **Fix:** Enable server-side encryption (SSE) with AWS KMS for the S3 bucket to ensure data is encrypted    │
│  at rest.                                                                                                       │
│                                                                                                                 │
│  - **Issue:** There is no encryption in transit for communications between Lambda, S3, and Bedrock.             │
│    - **Impact:** Data transmitted between these services is not encrypted, potentially exposing sensitive       │
│  information to interception.                                                                                   │
│    - **Fix:** Ensure that all communications use HTTPS and enable encryption in transit for S3 and Bedrock      │
│  interactions.                                                                                                  │
│                                                                                                                 │
│  - **Issue:** All Lambda functions share one IAM role, which has excessive permissions.                         │
│    - **Impact:** This increases the attack surface and

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 


**SECURITY:**

- **Issue:** The Lambda functions use an IAM role with `AdministratorAccess`.
  - **Impact:** This violates the principle of least privilege, allowing Lambda functions unnecessary broad access across the AWS account, increasing the risk of unauthorized actions and potential data breaches.
  - **Fix:** Create a custom IAM role for the Lambda functions with minimal required permissions, such as `s3:GetObject` for the specific S3 bucket, `bedrock:InvokeModel` for Claude on Bedrock, and necessary logging permissions.

- **Issue:** The S3 bucket containing HR documents is set to public-read.
  - **Impact:** This exposes sensitive HR documents to anyone on the internet, leading to potential data breaches and non-compliance with data protection regulations.
  - **Fix:** Remove public access to the S3 bucket. Use bucket policies to restrict access to authorized users only, and consider using pre-signed URLs for temporary access if needed.

- **Issue:** There is no encryption at

Watch the verbose log: the manager reads the task, picks a specialist, passes the
work, receives the answer, and moves on. You never wrote that routing logic.

**The trade-off:** a hierarchical crew makes many more model calls than a sequential one,
because the manager reasons before and after every delegation. Use sequential when you already
know the workflow; use hierarchical when you genuinely do not.

> **Worth doing with this output.** Check whether the crew found the problems you planted, and
> which ones it missed. A crew that produces three confident sections while missing the public
> S3 bucket is a good demonstration of why this pattern needs a human as the last step. The
> review is a first pass that makes a human reviewer faster — not a substitute for one.

---
## 5. Structured output

`output_pydantic` forces a task's result into a schema — which is what you need when the crew
feeds a downstream system rather than a human.

In [25]:
from pydantic import BaseModel, Field
from typing import List


class LaunchVerdict(BaseModel):
    recommendation: str = Field(description="One of: LAUNCH, LAUNCH_WITH_CHANGES, DO_NOT_LAUNCH")
    blocking_issues: List[str] = Field(description="Between 2 and 5 issues that must be fixed first")
    phi_exposure_risk: str = Field(description="One of: none, low, moderate, high")
    first_action: str = Field(description="The single highest-priority next step")


verdict_task = Task(
    description=(
        "Give a final launch verdict on the member-facing benefits chatbot reviewed earlier. "
        "Assume 40,000 members and a launch during open enrolment."
    ),
    expected_output="A structured verdict object.",
    agent=privacy_reviewer,
    output_pydantic=LaunchVerdict,
)

verdict_crew = Crew(agents=[privacy_reviewer], tasks=[verdict_task], verbose=False)
verdict_result = verdict_crew.kickoff()

parsed = verdict_result.pydantic
print("Recommendation   :", parsed.recommendation)
print("PHI exposure risk:", parsed.phi_exposure_risk)
print("First action     :", parsed.first_action)
print("Blocking issues  :")
for issue in parsed.blocking_issues:
    print("   -", issue)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Privacy and Compliance Reviewer                                                                         │
│                                                                                                                 │
│  Task: Give a final launch verdict on the member-facing benefits chatbot reviewed earlier. Assume 40,000        │
│  members and a launch during open enrolment.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Privacy and Compliance Reviewer                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "recommendation": "Conditional approval with immediate remediation required",                                │
│    "blocking_issues": [                                                                                         │
│      "Insufficient encryption at rest for stored PHI. Current implementation uses basic encryption which does   │
│  not meet regulatory standards.",                                                                               │
│      "Lack of business associate agreements (BAAs) with third-party vendors involved in data processing and     │
│  storage.",                                                                                                     │
│      "Inadequate audit logging mechanisms to track access and modifications to PHI.",                           │
│      "No retention limits defined for chat logs containing PHI, leading to potential long-term exposure         │
│  risks."                                                                                                        │
│    ],                                                                                                           │
│    "phi_exposure_risk": "High. The current system architecture poses significant risks of PHI exposure due to   │
│  the identified gaps in encryption, BAAs, audit logging, and data retention policies.",                         │
│    "first_action": "Implement robust encryption at rest for all stored PHI and ensure it complies with          │
│  regulatory standards. Simultaneously, draft and execute BAAs with all third-party vendors. Establish           │
│  comprehensive audit logging for all access and modifications to PHI and define clear retention limits for      │
│  chat logs."                                                                                                    │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

Recommendation   : Conditional approval with immediate remediation required
PHI exposure risk: High. The current system architecture poses significant risks of PHI exposure due to the identified gaps in encryption, BAAs, audit logging, and data retention policies.
First action     : Implement robust encryption at rest for all stored PHI and ensure it complies with regulatory standards. Simultaneously, draft and execute BAAs with all third-party vendors. Establish comprehensive audit logging for all access and modifications to PHI and define clear retention limits for chat logs.
Blocking issues  :
   - Insufficient encryption at rest for stored PHI. Current implementation uses basic encryption which does not meet regulatory standards.
   - Lack of business associate agreements (BAAs) with third-party vendors involved in data processing and storage.
   - Inadequate audit logging mechanisms to track access and modifications to PHI.
   - No retention limits defined for chat logs containing

---
## 6. Writing crews that actually work

Most disappointing crew output traces back to one of these five things:

| Problem | Fix |
|---|---|
| Agents produce generic text | The `backstory` is bland. Give it opinions, standards, and a method |
| Output is the wrong shape | `expected_output` is vague. Describe the format explicitly |
| Later agents ignore earlier work | You forgot `context=[earlier_task]` |
| The crew is slow and expensive | Too many agents. Three focused agents beat seven vague ones |
| Agents talk in circles | `allow_delegation=True` everywhere. Turn it off unless delegation is the point |

A rule that holds up well: **one agent per distinct skill, not per step**. If two steps need
the same expertise, that is one agent doing two tasks.

#### 6.1. One more rule for this domain

**Agreement is not verification.** A crew of three agents that all approve a member letter has
not checked it three times — they have generated three plausible approvals from the same
underlying model. Where correctness genuinely matters, verify with something that is not a
language model: a tool that computes the number, a retrieval step that cites the source, a
human who signs it off. A crew is a drafting and review accelerator, not an assurance
mechanism.

---
## Summary

- CrewAI models work as a **team of role-playing agents**
- `role` / `goal` / `backstory` shape behaviour; `backstory` matters most
- `expected_output` controls the deliverable — be specific
- `context=[...]` passes one task's output into the next
- `Process.sequential` for known workflows, `Process.hierarchical` for delegation
- Tools let agents act; `output_pydantic` makes results machine-readable
- Crews are ungrounded by default — attach tools and retrieval, or you get fluent guesses
- In this domain, put a human at the end of every crew

**Next:** notebook `6-Framework-Comparison.ipynb`, where we build the same application in all
three frameworks and compare them directly.